# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakeshkumarkhatri/flyrank-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

print("Token loaded:", HF_TOKEN is not None)
print("DuckDB connected to Hugging Face")

Token loaded: True
DuckDB connected to Hugging Face


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [7]:
import pandas as pd
import matplotlib.pyplot as plt

# March and April performance by content item
march = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available
    GROUP BY client_hash_id, content_hash_id
""").df()

april = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available
    GROUP BY client_hash_id, content_hash_id
""").df()

signal_df = march.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

signal_df["impression_change"] = (
    signal_df["april_impressions"] - signal_df["march_impressions"]
)

print("DISTRIBUTIONS")
print("=" * 50)
print(f"Rows: {len(signal_df):,}")
print(f"March impressions median: {signal_df['march_impressions'].median():.1f}")
print(f"March impressions mean: {signal_df['march_impressions'].mean():.1f}")
print(f"April impressions median: {signal_df['april_impressions'].median():.1f}")
print(f"April impressions mean: {signal_df['april_impressions'].mean():.1f}")

display(
    signal_df[
        ["march_impressions", "march_clicks",
         "april_impressions", "april_clicks",
         "impression_change"]
    ].describe()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DISTRIBUTIONS
Rows: 158,549
March impressions median: 246.0
March impressions mean: 1768.0
April impressions median: 200.0
April impressions mean: 1777.7


,march_impressions,march_clicks,april_impressions,april_clicks,impression_change
count,158549.000000,158549.000000,158549.000000,158549.000000,158549.000000
mean,1768.034671,5.176721,1777.749194,4.991176,9.714524
std,5706.756831,28.165721,6244.786008,31.113904,3124.504996
min,1.000000,0.000000,1.000000,0.000000,-162069.000000
25%,40.000000,0.000000,33.000000,0.000000,-191.000000
50%,246.000000,0.000000,200.000000,0.000000,-11.000000
75%,1264.000000,2.000000,1118.000000,2.000000,39.000000
max,617124.000000,5668.000000,799358.000000,7434.000000,261817.000000


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [8]:
# Signal test #1:
# Do low-impression pages show larger positive movement?

signal_df["march_impression_bucket"] = pd.cut(
    signal_df["march_impressions"],
    bins=[0, 9, 49, 199, float("inf")],
    labels=["1–9", "10–49", "50–199", "200+"]
)

signal_1 = (
    signal_df
    .groupby("march_impression_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        avg_march_impressions=("march_impressions", "mean"),
        avg_april_impressions=("april_impressions", "mean"),
        avg_change=("impression_change", "mean")
    )
    .reset_index()
)

display(signal_1)

,march_impression_bucket,n,avg_march_impressions,avg_april_impressions,avg_change
0,1–9,19258,4.038478,34.675875,30.637397
1,10–49,24298,25.763520,86.860359,61.096839
2,50–199,30417,111.223987,179.416741,68.192754
3,200+,84576,3266.095039,3235.252518,-30.842520


In [9]:
# Signal test #2:
# Does content age relate to future impression movement?

content_age = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
""").df()

content_age["content_created_date"] = pd.to_datetime(
    content_age["content_created_date"]
)

content_age["content_age_days"] = (
    pd.Timestamp("2026-03-31") -
    content_age["content_created_date"]
).dt.days

signal_df_age = signal_df.merge(
    content_age[
        ["client_hash_id", "content_hash_id", "content_age_days"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

signal_df_age["age_bucket"] = pd.cut(
    signal_df_age["content_age_days"],
    bins=[-1, 89, 179, 364, float("inf")],
    labels=["<90 days", "90–179", "180–364", "365+"]
)

signal_2 = (
    signal_df_age
    .groupby("age_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        avg_age_days=("content_age_days", "mean"),
        avg_march_impressions=("march_impressions", "mean"),
        avg_april_impressions=("april_impressions", "mean"),
        avg_change=("impression_change", "mean")
    )
    .reset_index()
)

display(signal_2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,age_bucket,n,avg_age_days,avg_march_impressions,avg_april_impressions,avg_change
0,<90 days,53907,47.441408,1483.932142,1867.909233,383.977090
1,90–179,24139,133.316997,2350.026969,1873.480757,-476.546212
2,180–364,60119,248.850131,1827.181157,1713.670653,-113.510504
3,365+,20384,408.205161,1655.719878,1614.936028,-40.783850


In [10]:
# Signal test #3:
# Does existing engagement (March clicks) relate to future movement?

signal_df["march_click_bucket"] = pd.cut(
    signal_df["march_clicks"],
    bins=[-1, 0, 1, 4, float("inf")],
    labels=["0", "1", "2–4", "5+"]
)

signal_3 = (
    signal_df
    .groupby("march_click_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        avg_march_clicks=("march_clicks", "mean"),
        avg_march_impressions=("march_impressions", "mean"),
        avg_april_impressions=("april_impressions", "mean"),
        avg_change=("impression_change", "mean")
    )
    .reset_index()
)

display(signal_3)

,march_click_bucket,n,avg_march_clicks,avg_march_impressions,avg_april_impressions,avg_change
0,0,90509,0.000000,240.101393,241.377907,1.276514
1,1,20045,1.000000,814.706810,744.629583,-70.077226
2,2–4,19193,2.723285,1760.368155,1645.828323,-114.539832
3,5+,28802,25.986077,7238.082355,7412.644990,174.562635


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [11]:
# Flag-linked test:
# Does the Week-4 "stale content" assumption have support?

flag_test = (
    signal_df_age
    .groupby("age_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        avg_march_impressions=("march_impressions", "mean"),
        avg_april_impressions=("april_impressions", "mean"),
        avg_change=("impression_change", "mean")
    )
    .reset_index()
)

display(flag_test)

print("\nFLAG-LINKED VERDICT")
print("Older content (365+) shows weaker average movement than newer content.")
print("Verdict: CONFIRMED directionally, but this is observational evidence, not causation.")

,age_bucket,n,avg_march_impressions,avg_april_impressions,avg_change
0,<90 days,53907,1483.932142,1867.909233,383.977090
1,90–179,24139,2350.026969,1873.480757,-476.546212
2,180–364,60119,1827.181157,1713.670653,-113.510504
3,365+,20384,1655.719878,1614.936028,-40.783850



FLAG-LINKED VERDICT
Older content (365+) shows weaker average movement than newer content.
Verdict: CONFIRMED directionally, but this is observational evidence, not causation.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [12]:
print(
    "The data supports using March impressions and content age as practical "
    "screening signals for human review."
)

print(
    "Low-volume pages and older content can be useful candidates for review, "
    "but the signals are directional rather than guarantees of future improvement."
)

print(
    "A content team should use these signals to prioritize investigation, "
    "then apply human judgment before making changes."
)

The data supports using March impressions and content age as practical screening signals for human review.
Low-volume pages and older content can be useful candidates for review, but the signals are directional rather than guarantees of future improvement.
A content team should use these signals to prioritize investigation, then apply human judgment before making changes.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.